In [2]:
!pip install bertopic
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from bertopic import BERTopic
import re



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 5.1 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/hdbscan/robust_single_linkage_.py:175: SyntaxWarning: invalid escape sequence '\{'
  $max \{ core_k(a), core_k(b), 1/\alpha d(a,b) \}$.


In [3]:
df = pd.read_csv("quran_sahih.csv")
docs = df["text"].astype(str).tolist()

def clean_english(text):
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

docs_clean = [clean_english(t) for t in docs]   

FileNotFoundError: [Errno 2] No such file or directory: 'quran_sahih.csv'

In [6]:
from google.colab import files
uploaded = files.upload()


: 

In [ ]:
from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

embedding_model = SentenceTransformer("all-mpnet-base-v2")

quran_stopwords = [
    "o", "ye", "thou", "thee", "thy", "shall",
    "verily", "lo", "behold", "indeed",
    "say", "said", "we", "they", "you"
]

vectorizer = CountVectorizer(stop_words=quran_stopwords)

topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer,      # ← correct place for stopwords
    calculate_probabilities=True,
    verbose=True,
    nr_topics = 10
)

topics, probs = topic_model.fit_transform(docs_clean)

topic_model.get_topic_info()
topic_model.visualize_hierarchy()


In [ ]:
topic_info = topic_model.get_topic_info()

for _, row in topic_info.iterrows():
    topic_id = row["Topic"]
    if topic_id == -1:
        continue  # skip outliers

    # Get top words for this topic
    top_words = topic_model.get_topic(topic_id)
    # Format as "word (weight)" and round weights
    formatted_words = [f"{word}: {weight:.4f}" for word, weight in top_words]

    # Print nicely
    print(f"\n===== Topic {topic_id} =====")
    print("Name:", row.get("Name", "N/A"))  # in case "Name" column is missing
    print("Top Words:")
    print(", ".join(formatted_words))
